In [ ]:
import pandas as pd
import numpy as np
import os, json, warnings
warnings.filterwarnings('ignore')

import matplotlib.font_manager as fm
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import optuna
import joblib

ARIAL_PATH = '/kaggle/input/datasets/hammaadali/arial-font/arial.ttf'
VIF_DIR = '/kaggle/working/VIF'

VIF_FILES = {
    'AGC': os.path.join(VIF_DIR, 'vif_final_AGC.csv'),
    'BGC': os.path.join(VIF_DIR, 'vif_final_BGC.csv'),
    'SOC0_30': os.path.join(VIF_DIR, 'vif_final_SOC0_30.csv'),
    'SOC0_100': os.path.join(VIF_DIR, 'vif_final_SOC0_100.csv'),
    'SOC0_200': os.path.join(VIF_DIR, 'vif_final_SOC0_200.csv'),
    'TECS': os.path.join(VIF_DIR, 'vif_final_TECS.csv'),
}

DATASETS = {
    'AGC': {'input_file': '/kaggle/working/Mangrove_AGC_clean_full.csv', 'response': 'AGC'},
    'BGC': {'input_file': '/kaggle/working/Mangrove_BGC_clean_full.csv', 'response': 'BGC'},
    'SOC0_30': {'input_file': '/kaggle/working/Mangrove_SOC0_30_clean_full.csv', 'response': 'SOC0_30'},
    'SOC0_100': {'input_file': '/kaggle/working/Mangrove_SOC0_100_clean_full.csv', 'response': 'SOC0_100'},
    'SOC0_200': {'input_file': '/kaggle/working/Mangrove_SOC0_200_clean_full.csv', 'response': 'SOC0_200'},
    'TECS': {'input_file': '/kaggle/working/Mangrove_TECS_clean_full.csv', 'response': 'TECS'},
}

OUTPUT_BASE_DIR = '/kaggle/working/ML_Results_NestedCV'
NODATA_VALUE = -9999
OUTER_FOLDS = 10
OUTER_REPETITIONS = 1
INNER_FOLDS = 5
OPTUNA_TRIALS = 30
RANDOM_SEED_FINAL = 42
LOG_TRANSFORM = False

if os.path.exists(ARIAL_PATH):
    fm.fontManager.addfont(ARIAL_PATH)
    font_prop = fm.FontProperties(fname=ARIAL_PATH)
    font_family = font_prop.get_name()
else:
    font_family = 'sans-serif'


def load_vif_predictors(vif_csv_path, tag):
    if not os.path.exists(vif_csv_path):
        raise FileNotFoundError(
            f"VIF file not found for '{tag}': {vif_csv_path}\n"
            f"Files in VIF dir: {os.listdir(VIF_DIR)}"
        )
    vif_df = pd.read_csv(vif_csv_path)
    predictors = vif_df['predictor'].tolist()
    return predictors


for tag, vif_path in VIF_FILES.items():
    try:
        DATASETS[tag]['predictors'] = load_vif_predictors(vif_path, tag)
    except FileNotFoundError:
        pass


def calc_metrics(y_true, y_pred):
    y_true, y_pred = np.array(y_true, dtype=np.float64), np.array(y_pred, dtype=np.float64)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    sd = np.std(y_true)
    iqr = np.percentile(y_true, 75) - np.percentile(y_true, 25)
    rpd = sd / rmse if rmse > 0 else 0.0
    rpiq = iqr / rmse if rmse > 0 else 0.0
    return dict(rmse=rmse, r2=r2, mae=mae, rpd=rpd, rpiq=rpiq)


def apply_log(y: np.ndarray) -> np.ndarray:
    return np.log1p(y.astype(np.float64)).astype(np.float32)


def undo_log(y: np.ndarray) -> np.ndarray:
    return np.maximum(np.expm1(y.astype(np.float64)), 0.0).astype(np.float64)


def load_dataset(cfg):
    df = pd.read_csv(cfg['input_file'], sep=',').replace(NODATA_VALUE, np.nan)
    response = cfg['response']
    df = df.dropna(subset=[response])

    available_num = [c for c in cfg['predictors'] if c in df.columns]

    keep = [response] + available_num
    df = df[keep].copy()
    for col in available_num:
        if df[col].isnull().any():
            df[col] = df[col].fillna(df[col].median())

    return df, available_num


def make_rf_objective(X_np: np.ndarray, y_np: np.ndarray):
    y_fit = apply_log(y_np) if LOG_TRANSFORM else y_np.astype(np.float32)

    def objective(trial):
        params = dict(
            n_estimators=trial.suggest_int('n_estimators', 100, 600),
            max_depth=trial.suggest_int('max_depth', 5, 30),
            min_samples_split=trial.suggest_int('min_samples_split', 2, 15),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 8),
            max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            random_state=42,
            n_jobs=-1,
        )
        model = RandomForestRegressor(**params)
        inner_kf = KFold(n_splits=INNER_FOLDS, shuffle=True, random_state=0)
        scores = []
        for tr, va in inner_kf.split(X_np):
            sc = StandardScaler()
            Xtr = sc.fit_transform(X_np[tr])
            Xva = sc.transform(X_np[va])
            model.fit(Xtr, y_fit[tr])
            scores.append(np.sqrt(mean_squared_error(y_fit[va], model.predict(Xva))))
        return np.mean(scores)

    return objective


def tune_on_fold(X_tr_np, y_tr_np, seed, label):
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(
        make_rf_objective(X_tr_np, y_tr_np),
        n_trials=OPTUNA_TRIALS,
        show_progress_bar=False,
    )
    return {**study.best_params, 'random_state': 42, 'n_jobs': -1}, study.best_value


def nested_cv(X_df, y_series, outer_folds, outer_reps):
    all_fold_metrics = []
    all_fold_params = []
    all_pred, all_true = [], []

    for rep in range(outer_reps):
        kf = KFold(n_splits=outer_folds, shuffle=True, random_state=rep)
        for fold, (tr_idx, va_idx) in enumerate(kf.split(X_df), 1):
            X_tr_raw = X_df.iloc[tr_idx].values
            X_va_raw = X_df.iloc[va_idx].values
            y_tr_orig = y_series.iloc[tr_idx].values.astype(np.float64)
            y_va_orig = y_series.iloc[va_idx].values.astype(np.float64)

            best_params, inner_score = tune_on_fold(
                X_tr_raw, y_tr_orig, seed=rep * 1000 + fold, label=f"rep{rep + 1}_fold{fold}"
            )

            sc = StandardScaler()
            X_tr = sc.fit_transform(X_tr_raw)
            X_va = sc.transform(X_va_raw)

            y_tr_fit = apply_log(y_tr_orig) if LOG_TRANSFORM else y_tr_orig

            model = RandomForestRegressor(**best_params)
            model.fit(X_tr, y_tr_fit)

            preds_raw = model.predict(X_va)
            preds = undo_log(preds_raw) if LOG_TRANSFORM else preds_raw

            m = calc_metrics(y_va_orig, preds)
            m.update({'rep': rep + 1, 'fold': fold, 'inner_score': inner_score})
            all_fold_metrics.append(m)
            all_fold_params.append({'rep': rep + 1, 'fold': fold, **best_params})
            all_pred.extend(preds.tolist())
            all_true.extend(y_va_orig.tolist())

    metrics_keys = ['rmse', 'r2', 'mae', 'rpd', 'rpiq']
    mean_m = {k: np.mean([f[k] for f in all_fold_metrics]) for k in metrics_keys}
    std_m = {f'{k}_std': np.std([f[k] for f in all_fold_metrics]) for k in metrics_keys}
    overall = calc_metrics(all_true, all_pred)

    return {
        'fold_metrics': all_fold_metrics,
        'fold_params': all_fold_params,
        'mean': mean_m,
        'std': std_m,
        'overall': overall,
    }


def run_one(tag, cfg, out_dir):
    if 'predictors' not in cfg:
        return None, None

    df, feature_cols = load_dataset(cfg)
    X = df[feature_cols]
    y = df[cfg['response']]

    nested = nested_cv(X, y, OUTER_FOLDS, OUTER_REPETITIONS)

    final_best_params, _ = tune_on_fold(
        X.values, y.values, seed=RANDOM_SEED_FINAL, label="final full-data fit"
    )
    sc_final = StandardScaler()
    X_scaled = sc_final.fit_transform(X.values)
    y_fit_final = apply_log(y.values) if LOG_TRANSFORM else y.values
    final_model = RandomForestRegressor(**final_best_params)
    final_model.fit(X_scaled, y_fit_final)

    feat_imp = sorted(
        zip(feature_cols, final_model.feature_importances_),
        key=lambda x: x[1], reverse=True,
    )

    os.makedirs(out_dir, exist_ok=True)

    pd.DataFrame(nested['fold_metrics']).to_csv(
        os.path.join(out_dir, f'nested_fold_metrics_{tag}.csv'), index=False)

    pd.DataFrame(nested['fold_params']).to_csv(
        os.path.join(out_dir, f'nested_fold_params_{tag}.csv'), index=False)

    pd.DataFrame(feat_imp, columns=['Feature', 'Importance']).to_csv(
        os.path.join(out_dir, f'feature_importance_{tag}.csv'), index=False)

    summary = {
        'tag': tag,
        'response': cfg['response'],
        'log_transform': LOG_TRANSFORM,
        'n_obs': len(df),
        'n_features': len(feature_cols),
        'features_used': feature_cols,
        'outer_folds': OUTER_FOLDS,
        'outer_reps': OUTER_REPETITIONS,
        'inner_folds': INNER_FOLDS,
        'optuna_trials': OPTUNA_TRIALS,
        'final_best_params': {k: (v if not isinstance(v, np.generic) else v.item())
                               for k, v in final_best_params.items()},
        'mean_r2': round(nested['mean']['r2'], 4),
        'std_r2': round(nested['std']['r2_std'], 4),
        'mean_rmse': round(nested['mean']['rmse'], 4),
        'std_rmse': round(nested['std']['rmse_std'], 4),
        'mean_mae': round(nested['mean']['mae'], 4),
        'std_mae': round(nested['std']['mae_std'], 4),
        'mean_rpd': round(nested['mean']['rpd'], 4),
        'std_rpd': round(nested['std']['rpd_std'], 4),
        'mean_rpiq': round(nested['mean']['rpiq'], 4),
        'std_rpiq': round(nested['std']['rpiq_std'], 4),
        'overall_r2': round(nested['overall']['r2'], 4),
        'overall_rmse': round(nested['overall']['rmse'], 4),
        'overall_mae': round(nested['overall']['mae'], 4),
        'overall_rpd': round(nested['overall']['rpd'], 4),
        'overall_rpiq': round(nested['overall']['rpiq'], 4),
    }
    with open(os.path.join(out_dir, f'summary_{tag}.json'), 'w') as f:
        json.dump(summary, f, indent=4)

    joblib.dump(
        {
            'model': final_model,
            'scaler': sc_final,
            'features': feature_cols,
            'log_transform': LOG_TRANSFORM,
        },
        os.path.join(out_dir, f'rf_model_{tag}.pkl'),
    )

    return summary, nested


os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
all_summaries = []

for tag, cfg in DATASETS.items():
    out_dir = os.path.join(OUTPUT_BASE_DIR, tag)
    summary, nested = run_one(tag, cfg, out_dir)
    if summary is not None:
        all_summaries.append(summary)

if all_summaries:
    cols_order = [
        'tag', 'response', 'log_transform', 'n_obs', 'n_features',
        'outer_folds', 'outer_reps', 'inner_folds', 'optuna_trials',
        'mean_r2', 'std_r2',
        'mean_rmse', 'std_rmse',
        'mean_mae', 'std_mae',
        'mean_rpd', 'std_rpd',
        'mean_rpiq', 'std_rpiq',
        'overall_r2', 'overall_rmse', 'overall_mae',
        'overall_rpd', 'overall_rpiq',
    ]
    summary_df = pd.DataFrame(all_summaries)[cols_order]
    summary_csv = os.path.join(OUTPUT_BASE_DIR, 'summary_all_nested.csv')
    summary_df.to_csv(summary_csv, index=False)